# Math Modal fine-tuning: Colab quickstart

This Colab quickstart installs and runs the package in this repository. Use a Colab runtime with a GPU (Runtime ? Change runtime type ? T4 GPU or another supported NVIDIA GPU). Model downloads are large and training takes time.

Before running the setup cell, set `REPO_URL` to your public GitHub repository URL.

In [ ]:
from pathlib import Path
import subprocess

REPO_URL = input("Enter the HTTPS clone URL of your public GitHub repository: ").strip()
REPO_DIR = Path("/content/math-modal-finetuning")

if not REPO_URL:
    raise ValueError("A public GitHub repository URL is required.")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

print(f"Repository: {REPO_DIR}")

## Install the project

This follows the installation commands in the project README. Dependency versions are not pinned, and Colab/package compatibility can change over time.

In [ ]:
%cd /content/math-modal-finetuning
%pip install -r requirements.txt
%pip install -e .

## Configure paths

The default output matches the CLI and README: `./openai-gsm8k` inside the cloned project. To save directly to Google Drive, mount Drive and set `OUTPUT_DIR` to `/content/drive/MyDrive/openai-gsm8k`.

In [ ]:
from pathlib import Path
import subprocess

PROJECT_DIR = Path("/content/math-modal-finetuning")
OUTPUT_DIR = "./openai-gsm8k"

# Optional Drive-backed output. Uncomment to mount Drive and save checkpoints there.
# from google.colab import drive
# drive.mount("/content/drive")
# OUTPUT_DIR = "/content/drive/MyDrive/openai-gsm8k"

def run_cli(*args):
    command = ["math-modal", *map(str, args)]
    print("$", " ".join(command))
    subprocess.run(command, cwd=PROJECT_DIR, check=True)

## Train

This runs the configured GSM8K subset and training workflow. It downloads the base model and dataset. The output path is configurable; checkpoints and the final adapter/tokenizer are written there.

In [ ]:
run_cli("train", "--output-dir", OUTPUT_DIR)

## Run inference

The prompt and generation call use the package inference implementation. Change `QUESTION` and `MAX_NEW_TOKENS` as needed; example configurations use token limits of 50, 500, and 1000.

In [ ]:
QUESTION = "Find the value of x if 3x + 7 = 22."
MAX_NEW_TOKENS = 50

run_cli(
    "infer",
    "--model-path", OUTPUT_DIR,
    "--question", QUESTION,
    "--max-new-tokens", MAX_NEW_TOKENS,
)

## Evaluate

This compares the base model and adapter on the same seeded sample of the GSM8K test split and writes results to `gsm8k_evaluation/`. It requires the adapter produced by training.

In [ ]:
run_cli(
    "evaluate",
    "--model-path", OUTPUT_DIR,
    "--n-examples", 200,
    "--seed", 42,
    "--max-new-tokens", 512,
)

## Notes

- The CLI uses 4-bit bitsandbytes quantization and expects a compatible NVIDIA GPU environment. CPU compatibility is not claimed.
- Google Drive mounting is optional and must be enabled explicitly in the path setup above.
- Do not commit model weights, credentials, or private datasets to the public repository.